# SentinelAI — 05 1D CNN (Validation Only)

هذه CNN تعمل على محور الخصائص الـ77 كـ1D sequence لأغراض متطلب المقرر.
هي **ليست time-series CNN** لأن ترتيب الخصائص ليس محورًا زمنيًا.

التدريب على Train، وEarly Stopping/التقييم على Validation فقط. لا يتم تحميل Test هنا.

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

cwd = Path.cwd()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd
P = BASE_DIR / "data" / "processed"
M = BASE_DIR / "models"
M.mkdir(exist_ok=True)

X_train = np.load(P / "X_train.npy")
X_val = np.load(P / "X_val.npy")
y_train = np.load(P / "y_train.npy")
y_val = np.load(P / "y_val.npy")

label_mapping = {int(k): v for k, v in json.loads((P/"label_mapping.json").read_text(encoding="utf-8")).items()}
class_weights = {int(k): v for k, v in json.loads((P/"class_weights.json").read_text(encoding="utf-8")).items()}

le = LabelEncoder().fit(y_train)
y_train_enc = le.transform(y_train)
y_val_enc = le.transform(y_val)

class_weight_enc = {int(i): float(class_weights[int(c)]) for i, c in enumerate(le.classes_)}

X_train_cnn = X_train.reshape(-1, X_train.shape[1], 1)
X_val_cnn = X_val.reshape(-1, X_val.shape[1], 1)

print("Train:", X_train_cnn.shape, "Validation:", X_val_cnn.shape)

Train: (908327, 77, 1) Validation: (113541, 77, 1)


In [2]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1], 1)),
    layers.Conv1D(64, kernel_size=3, activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(128, kernel_size=3, activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(len(le.classes_), activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 77, 64)              │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 77, 64)              │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 38, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 38, 128)             │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 38, 128)             │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ (None, 128)                 │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │          16,512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 11)                  │           1,419 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 43,659 (170.54 KB)

 Trainable params: 43,275 (169.04 KB)

 Non-trainable params: 384 (1.50 KB)

In [3]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5
    ),
]

start = time.time()
history = model.fit(
    X_train_cnn,
    y_train_enc,
    validation_data=(X_val_cnn, y_val_enc),
    epochs=15,
    batch_size=1024,
    class_weight=class_weight_enc,
    callbacks=callbacks,
    verbose=1,
)
train_seconds = time.time() - start

Epoch 1/15


  1/888 ━━━━━━━━━━━━━━━━━━━━ 24:01 2s/step - accuracy: 0.0576 - loss: 7.7315

  2/888 ━━━━━━━━━━━━━━━━━━━━ 1:37 110ms/step - accuracy: 0.0659 - loss: 4.8712

  3/888 ━━━━━━━━━━━━━━━━━━━━ 1:37 110ms/step - accuracy: 0.0723 - loss: 3.8409

  4/888 ━━━━━━━━━━━━━━━━━━━━ 1:38 111ms/step - accuracy: 0.0889 - loss: 3.2849

...

884/888 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.9487 - loss: 0.1886

885/888 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.9488 - loss: 0.1884

886/888 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.9488 - loss: 0.1883

887/888 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.9488 - loss: 0.1881

888/888 ━━━━━━━━━━━━━━━━━━━━ 126s 142ms/step - accuracy: 0.9488 - loss: 0.1881 - val_accuracy: 0.9846 - val_loss: 0.0854 - learning_rate: 2.5000e-04


In [4]:
proba = model.predict(X_val_cnn, batch_size=2048, verbose=0)
pred_enc = np.argmax(proba, axis=1)
pred = le.inverse_transform(pred_enc)

p, r, f1, _ = precision_recall_fscore_support(
    y_val, pred, average="macro", zero_division=0
)
acc = accuracy_score(y_val, pred)

print(f"Validation Accuracy: {acc:.6f}")
print(f"Validation Macro Precision: {p:.6f}")
print(f"Validation Macro Recall: {r:.6f}")
print(f"Validation Macro F1: {f1:.6f}")
print(classification_report(
    y_val, pred,
    labels=sorted(label_mapping),
    target_names=[label_mapping[c] for c in sorted(label_mapping)],
    zero_division=0,
))

Validation Accuracy: 0.984561
Validation Macro Precision: 0.626864
Validation Macro Recall: 0.936387
Validation Macro F1: 0.672114
              precision    recall  f1-score   support

      Benign       1.00      0.98      0.99     88006
    Attack_2       1.00      0.92      0.96      3513
    Attack_3       0.99      1.00      1.00      3381
    Attack_4       0.96      1.00      0.98     12428
    Attack_5       0.96      1.00      0.98      5205
    Attack_6       0.65      1.00      0.79       760
    Attack_7       0.37      0.96      0.54       203
    Attack_8       0.81      1.00      0.90        30
    Attack_9       0.03      0.44      0.06         9
   Attack_10       0.04      1.00      0.07         5
   Attack_11       0.07      1.00      0.13         1

    accuracy                           0.98    113541
   macro avg       0.63      0.94      0.67    113541
weighted avg       0.99      0.98      0.99    113541



In [5]:
model.save(M / "cnn_model.keras")
joblib.dump(le, M / "cnn_label_encoder.joblib")

history_json = {k: [float(v) for v in vals] for k, vals in history.history.items()}
(M / "cnn_history.json").write_text(
    json.dumps(history_json, indent=2), encoding="utf-8"
)

pd.DataFrame([{
    "Model": "CNN (1D Conv)",
    "Evaluation_Split": "validation",
    "Accuracy": acc,
    "Precision_macro": p,
    "Recall_macro": r,
    "F1_macro": f1,
    "Train_seconds": train_seconds,
}]).to_csv(M / "cnn_comparison.csv", index=False)

لا توجد أي نتيجة Test في هذا الملف. الاختبار النهائي محفوظ للملف 06 فقط.